# 第4章 优化算法 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter4/Chapter4.md]]（最速下降、线搜索、Momentum/NAG、矩阵AdaGrad、RMSProp、Adam、AdaMax、小批量）
- [[课程/深度学习/课堂笔记/2026-09-16-第6节-深度学习优化理论讲座.md]]（第四至八节与学习率边界校注）
- [[课程/深度学习/课堂笔记/2026-09-21-第7节-随机优化算法讲解.md]]（第二至五节；Chapter4(1).py程序解读）
- [[课程/深度学习/演示代码/Chapter4(1).py]]（d=5、SGD/Adam/RMSprop、1000步、初始全1）
- [[课程/深度学习/作业/Homework02/Homework02.md]]（Exercise 5、6、8；原参数下如实报告发散）
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]（C4-03—10、A-06、07、10）
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]（30分钟、100分及API范围）


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q4A 线性仪器校准：手动 Momentum 的完整优化实验

**30分钟，100分；依据第4章动量与MSE新编的稳定问题，不是Rosenbrock作业换参数。**

校准数据 `x=[-2,-1,0,1,2]`，`y=1.5*x-.5`；模型 $\hat y=a x+b$，$J(a,b)=\operatorname{mean}(\hat y-y)^2$。
从导入开始用NumPy手写完整程序，禁止torch.optim，初值theta=(0,0)、v=(0,0)，eta=.1、mu=.8，120步。

1. **25分/约7分钟**：创建数据，定义 `calibration_loss(theta)` 与手动解析梯度 `calibration_grad(theta)`，打印初始损失及梯度。
2. **35分/约10分钟**：定义 `train_calibration()`，按课件速度约定 `v=mu*v-eta*g,theta=theta+v`；返回最终theta、含0—120步的121项损失。保存为 `theta4a,history4a`，每一步在更新后重算损失，打印参数误差与最终损失。
3. **20分/约5分钟**：对probe=(.2,-.1)的两个参数做 $h=10^{-6}$ 中心差分，保存 `gradient_error4a`（最大绝对误差），打印解析/数值梯度。
4. **20分/约8分钟**：保存 `output/Q4A.png`：画迭代—损失半对数曲线（英文标签）。打印 `abs(theta4a-[1.5,-.5])`；说明某一步损失上升不必等同实现错误，不能要求动量损失逐步单调。

评分时既看结果，也检查速度状态独立保存与负梯度方向；无代码补全。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
x4a=np.array([-2.,-1.,0.,1.,2.]);y4a=1.5*x4a-.5
def calibration_loss(theta):
    return float(np.mean((theta[0]*x4a+theta[1]-y4a)**2))
def calibration_grad(theta):
    residual=theta[0]*x4a+theta[1]-y4a
    return np.array([2*np.mean(residual*x4a),2*np.mean(residual)])
def train_calibration():
    theta=np.zeros(2);v=np.zeros(2);history=[calibration_loss(theta)]
    for _ in range(120):
        g=calibration_grad(theta)
        v=.8*v-.1*g
        theta=theta+v
        history.append(calibration_loss(theta))
    return theta,np.array(history)
print('initial loss and gradient',calibration_loss(np.zeros(2)),calibration_grad(np.zeros(2)))
theta4a,history4a=train_calibration()
probe4a=np.array([.2,-.1]);h4a=1e-6
fd4a=[]
for j in range(2):
    xp=probe4a.copy();xm=probe4a.copy();xp[j]+=h4a;xm[j]-=h4a
    fd4a.append((calibration_loss(xp)-calibration_loss(xm))/(2*h4a))
fd4a=np.array(fd4a)
gradient_error4a=float(np.max(np.abs(fd4a-calibration_grad(probe4a))))
print('analytic/numeric/error',calibration_grad(probe4a),fd4a,gradient_error4a)
print('final theta',theta4a,'parameter error',np.abs(theta4a-[1.5,-.5]),'final loss',history4a[-1])
plt.figure();plt.semilogy(np.arange(121),np.maximum(history4a,1e-30),label='manual Momentum')
plt.xlabel('iteration');plt.ylabel('calibration MSE');plt.legend()
Path('output').mkdir(exist_ok=True);plt.savefig('output/Q4A.png');plt.close()


### 解析与点评

**评分点评**：25分检查MSE与梯度 `(-6,1)`；35分检查前两步动量、121项日志及末值重算；20分检查非最优probe上的两坐标差分；20分检查半对数图及参数误差解释。目标Hessian为diag(4,2)，是给定步长下可控的独立稳定题。仅检查最终收敛不足以判断真正使用了Momentum，故公开自检固定了第二步损失。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
assert abs(calibration_loss(np.zeros(2))-4.75)<1e-12
assert np.allclose(calibration_grad(np.zeros(2)),[-6.,1.])
assert np.allclose(calibration_grad(np.array([1.5,-.5])),[0.,0.])
_check_history4a = np.asarray(history4a, dtype=float)
assert _check_history4a.shape==(121,) and abs(_check_history4a[1]-1.78)<1e-12
# Momentum-specific second update: theta1=(.6,-.1), theta2=(1.44,-.26).
assert abs(_check_history4a[2]-(2*.06**2+.24**2))<1e-12
assert np.max(np.abs(np.asarray(theta4a)-[1.5,-.5]))<1e-5 and _check_history4a[-1]<1e-9
assert gradient_error4a<1e-7 and abs(_check_history4a[-1]-calibration_loss(theta4a))<1e-12
assert Path('output/Q4A.png').stat().st_size>1000

print("自检通过")
